In [0]:
-- ================================================================================================================
-- GHG EMISSIONS - GOLD LAYER
-- SQL ANALYTICS / BUSINESS QUESTIONS
-- ================================================================================================================
--
-- Purpose :
-- Use the curated Gold table to perform analytical SQL and answer business-oriented questions about GHG emissions.
--
-- Dataset grain :
-- One record represents one sector, one emission scope and one available year.
--
-- Main columns :
-- year : available observation year
-- emission_scope : Local or Outside Paris
-- sector : emissions sector
-- emissions_mt_co2 : emissions measured in Mt CO2
--
-- Assumed Gold table name
-- gold_ghg_emissions
--
-- Analytics objective
-- Move from data preparation to business analysis: understand
-- emission levels, compare scopes and sectors, identify trends,
-- measure changes over time and produce decision-support metrics.
-- ================================================================================================================

-- Display the gold_ghg_emissions 

SELECT *
FROM workspace.gold.gold_ghg_emissions ;

-- Total Emissions & Average Intensity by Sector

SELECT 
    sector,
    sum(emissions_mt_co2) as total_emissions,
    avg(emissions_mt_co2) as average_intensity
FROM  workspace.gold.gold_ghg_emissions
GROUP BY sector
ORDER BY total_emissions DESC

--  Top 5 Highest Emitting sectors 

SELECT 
    sector,
    sum(emissions_mt_co2) as total_emissions
FROM workspace.gold.gold_ghg_emissions
GROUP BY sector
ORDER BY total_emissions DESC
LIMIT 5;

--  all available years in chronological order.

SELECT DISTINCT year 
FROM workspace.gold.gold_ghg_emissions
ORDER BY year DESC;

--  every emission scope and sector combination available in the Gold dataset.

SELECT 
    DISTINCT emission_scope,
    sector
FROM workspace.gold.gold_ghg_emissions;

-- total GHG emissions across the complete dataset.

SELECT 
    ROUND(SUM (emissions_mt_co2),2) AS total_emissions
FROM workspace.gold.gold_ghg_emissions ;

-- total emissions by emission scope and sorting the scopes from highest to lowest total

SELECT 
    emission_scope ,
    ROUND(SUM(emissions_mt_co2),2) AS total_emissions
FROM workspace.gold.gold_ghg_emissions 
GROUP BY emission_scope
ORDER BY total_emissions DESC ; 

--  average annual emissions for every sector and sorting sectors from highest to lowest average

SELECT 
    sector,
    ROUND (AVG(emissions_mt_co2),2) AS average_year_emissions 
FROM workspace.gold.gold_ghg_emissions 
GROUP BY sector 
ORDER BY average_year_emissions DESC ; 

--  the sector with the highest total emissions across all available years.

SELECT
    sector,
    ROUND(SUM(emissions_mt_co2), 2) AS total_emissions_mt_co2
FROM workspace.gold.gold_ghg_emissions
GROUP BY sector
ORDER BY total_emissions_mt_co2 DESC
LIMIT 1;

--  the sector with the lowest average annual emissions.

SELECT 
    sector,
    ROUND(AVG(emissions_mt_co2),2) AS averrage
FROM workspace.gold.gold_ghg_emissions 
GROUP BY sector
ORDER BY averrage
LIMIT 1; 

--  total emissions for each year 

SELECT 
    year,
    ROUND(SUM(emissions_mt_co2),2) AS total_emissions
FROM workspace.gold.gold_ghg_emissions 
GROUP BY year 
ORDER BY year ; 

-- annual emissions separately for Local and Outside Paris scopes

SELECT 
    year,
    emission_scope,
    ROUND(SUM(emissions_mt_co2),2) AS total_emissions
FROM workspace.gold.gold_ghg_emissions 
GROUP BY year,emission_scope
ORDER BY year ;

--  the three sectors with the highest total emissions across the complete period

SELECT 
    sector,
    ROUND(SUM(emissions_mt_co2),2) AS total_emission,
    RANK() OVER (ORDER BY SUM(emissions_mt_co2) DESC) AS ranking
FROM workspace.gold.gold_ghg_emissions 
GROUP BY sector
LIMIT 3;

-- sectors whose average annual emissions are greater than the overall average emission value.

SELECT 
    sector,
    ROUND(AVG(emissions_mt_co2),2) AS avg
FROM  workspace.gold.gold_ghg_emissions 
GROUP BY sector
HAVING avg >(
    SELECT ROUND(AVG(emissions_mt_co2),2)
    FROM  workspace.gold.gold_ghg_emissions 
)
ORDER BY avg DESC; 

--  the percentage of total emissions represented by each emission scope.

WITH scope_totals AS (
    SELECT
        emission_scope,
        SUM(emissions_mt_co2) AS scope_emissions
    FROM workspace.gold.gold_ghg_emissions
    GROUP BY emission_scope
),
grand_total AS (
    SELECT SUM(emissions_mt_co2) AS total_emissions
    FROM workspace.gold.gold_ghg_emissions
)
SELECT
    s.emission_scope,
    ROUND(s.scope_emissions, 2) AS emissions_mt_co2,
    ROUND(100 * s.scope_emissions / g.total_emissions, 2) AS pct_of_total
FROM scope_totals s
CROSS JOIN grand_total g
ORDER BY pct_of_total DESC;

-- Calculate the year-over-year change in total emissions .

WITH yearly AS (
    SELECT
        year,
        SUM(emissions_mt_co2) AS total_emissions
    FROM workspace.gold.gold_ghg_emissions
    GROUP BY year
)
SELECT
    year,
    ROUND(total_emissions, 2) AS total_emissions_mt_co2,
    ROUND(LAG(total_emissions) OVER (ORDER BY year), 2) AS previous_year_emissions,
    ROUND(
        total_emissions - LAG(total_emissions) OVER (ORDER BY year),
        2
    ) AS change_mt_co2
FROM yearly
ORDER BY year;

-- the year-over-year percentage change in total emissions 

WITH yearly AS (
    SELECT
        year,
        SUM(emissions_mt_co2) AS total_emissions
    FROM workspace.gold.gold_ghg_emissions
    GROUP BY year
),
changes AS (
    SELECT
        year,
        total_emissions,
        LAG(total_emissions) OVER (ORDER BY year) AS previous_emissions
    FROM yearly
)
SELECT
    year,
    ROUND(total_emissions, 2) AS total_emissions_mt_co2,
    ROUND(previous_emissions, 2) AS previous_year_emissions,
    ROUND(
        100 * (total_emissions - previous_emissions) / previous_emissions,
        2
    ) AS yoy_change_pct
FROM changes
ORDER BY year;

-- highest-emitting sector inside each emission scope 

WITH sector_totals AS (
    SELECT
        emission_scope,
        sector,
        SUM(emissions_mt_co2) AS total_emissions
    FROM workspace.gold.gold_ghg_emissions
    GROUP BY emission_scope, sector
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY emission_scope
            ORDER BY total_emissions DESC
        ) AS rn
    FROM sector_totals
)
SELECT
    emission_scope,
    sector,
    ROUND(total_emissions, 2) AS total_emissions_mt_co2
FROM ranked
WHERE rn = 1
ORDER BY emission_scope;

-- the year with the highest recorded emissions for every sector 

WITH ranked AS (
    SELECT
        sector,
        year,
        emissions_mt_co2,
        ROW_NUMBER() OVER (
            PARTITION BY sector
            ORDER BY emissions_mt_co2 DESC, year DESC
        ) AS rn
    FROM workspace.gold.gold_ghg_emissions
)
SELECT
    sector,
    year,
    ROUND(emissions_mt_co2, 2) AS emissions_mt_co2
FROM ranked
WHERE rn = 1
ORDER BY emissions_mt_co2 DESC;
